# Data Import — Emerging Markets (Part 2)

Downloads competition data from Kaggle (iaa-ai-assisted-hackathon-2026-part-2) using `kagglehub`, extracting only the **emerging** market folder and saving to `data_emerging/`.

In [0]:
%pip install kagglehub

In [0]:
import os
import zipfile
import shutil
import kagglehub

# --- Kaggle credentials ---
# kagglehub supports the newer single-token auth via KAGGLE_API_TOKEN.
# Get your token from: kaggle.com → Account → API Token
# Set it as an environment variable before running this cell:
#   export KAGGLE_API_TOKEN=KGAT_xxxxx
# Or replace the placeholder below.

# NOTE: Insert your current Kaggle API token here before running.
# Get a fresh token from: kaggle.com → Account → API Token
# (Previous token removed for security — do not commit real tokens to git.)
os.environ["KAGGLE_API_TOKEN"] = os.environ.get("KAGGLE_API_TOKEN", "<your-kaggle-api-token>")

print("Kaggle credentials set.")

In [0]:
# Project paths
PROJECT_ROOT = "/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026"
DATA_EMERGING = os.path.join(PROJECT_ROOT, "data_emerging")

# Download the competition data (returns path to downloaded zip)
print("Downloading competition data from Kaggle...")
zip_path = kagglehub.competition_download("iaa-ai-assisted-hackathon-2026-part-2")
print(f"Downloaded to: {zip_path}")

# The download is a zip file — inspect its contents
if zip_path.endswith(".zip"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        all_files = zf.namelist()
        print(f"\nTotal files in zip: {len(all_files)}")
        # Show top-level structure
        top_dirs = set(f.split("/")[0] for f in all_files if "/" in f)
        print(f"Top-level folders: {top_dirs}")
        
        # Filter for emerging folder only
        emerging_files = [f for f in all_files if f.startswith("emerging/") or f.startswith("/emerging/")]
        print(f"\nEmerging files found: {len(emerging_files)}")
        for f in emerging_files:
            print(f"  {f}")
        
        # Extract only emerging files to data_emerging
        os.makedirs(DATA_EMERGING, exist_ok=True)
        for f in emerging_files:
            # Get the filename (strip the 'emerging/' prefix)
            filename = f.replace("emerging/", "").replace("/emerging/", "")
            if not filename or filename.endswith("/"):
                continue  # skip directories
            target_path = os.path.join(DATA_EMERGING, filename)
            with zf.open(f) as src, open(target_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"  Extracted: {filename}")
else:
    # If it's a directory, copy the emerging subfolder
    emerging_src = os.path.join(zip_path, "emerging")
    if os.path.isdir(emerging_src):
        os.makedirs(DATA_EMERGING, exist_ok=True)
        for f in os.listdir(emerging_src):
            shutil.copy2(os.path.join(emerging_src, f), os.path.join(DATA_EMERGING, f))
            print(f"  Copied: {f}")
    else:
        print(f"Unexpected path type: {zip_path}")
        print(f"Contents: {os.listdir(zip_path)}")

print(f"\nData saved to: {DATA_EMERGING}")

In [0]:
# Verify the downloaded files
print(f"Files in {DATA_EMERGING}:")
for f in sorted(os.listdir(DATA_EMERGING)):
    fpath = os.path.join(DATA_EMERGING, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.2f} MB)")

# Quick check — load each parquet file and show row count + columns
import pandas as pd

for f in sorted(os.listdir(DATA_EMERGING)):
    if f.endswith(".parquet"):
        fpath = os.path.join(DATA_EMERGING, f)
        df = pd.read_parquet(fpath)
        print(f"\n{f}: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"  Columns: {list(df.columns[:10])}{'...' if len(df.columns) > 10 else ''}")